In [ ]:
# ============================================================
# ADVANCED RESNET50 PIPELINE
# CHEST X-RAY PNEUMONIA CLASSIFICATION
# SMALL DATASET OPTIMIZED
# ============================================================

# ============================================================
# 1. INSTALLS
# ============================================================
!pip install opencv-python -q

# ============================================================
# 2. Imports
# ============================================================
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    f1_score
)

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# ============================================================
# 3. Config
# ============================================================
from google.colab import drive

drive.mount('/content/drive')

data_dir = '/content/drive/MyDrive/UFS - Mestrado/Visão Computacional/DataSets/chest ray-x pneumonia'

IMG_SIZE = (224,224)

BATCH_SIZE = 16

EPOCHS = 50

SEED = 42

DATASET_PERCENTAGE = 1

In [ ]:
# ============================================================
# 4. Load datasets
# ============================================================

# TRAIN
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir + "/train",
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# VALIDATION
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir + "/train",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# TEST
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir + "/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

In [ ]:
# ============================================================
# 6. Compute class weights
# ============================================================
y_train = []

for images, labels in train_ds:
    y_train.extend(labels.numpy().flatten())

y_train = np.array(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    0: class_weights[0],
    1: class_weights[1]
}

print("\nCLASS WEIGHTS")
print(class_weights)

In [ ]:
# ============================================================
# 7. Performance optimization
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# ============================================================
# 8. Data augmentation
# ============================================================
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.08),

    layers.RandomZoom(0.08),

    layers.RandomContrast(0.1)

])

In [ ]:
# ============================================================
# 9. Build ResNet50
# ============================================================
def build_resnet50():

    # ========================================================
    # Backbone
    # ========================================================
    base_model = ResNet50(

        include_top=False,

        weights='imagenet',

        input_shape=(224,224,3)
    )

    # ========================================================
    # Fine tuning
    # ========================================================
    base_model.trainable = True

    for layer in base_model.layers[:-120]:
        layer.trainable = False

    # ========================================================
    # Input
    # ========================================================
    inputs = tf.keras.Input(
        shape=(224,224,3)
    )

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.resnet.preprocess_input(x)

    # Backbone
    x = base_model(
        x,
        training=False
    )

    # ========================================================
    # Classification head
    # ========================================================
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(0.4)(x)

    x = layers.Dense(
        256,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    )(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        1,
        activation='sigmoid'
    )(x)

    model = tf.keras.Model(
        inputs,
        outputs
    )

    return model

# ============================================================
# 10. Create model
# ============================================================
model = build_resnet50()

model.summary()

In [ ]:
# ============================================================
# 11. Cosine decay LR
# ============================================================
steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(

    initial_learning_rate=1e-4,

    decay_steps=steps_per_epoch * EPOCHS
)

In [ ]:
# ============================================================
# 12. Compile
# ============================================================
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=lr_schedule
    ),

    loss=tf.keras.losses.BinaryFocalCrossentropy(

        gamma=2.0,

        apply_class_balancing=True
    ),

    metrics=[

        'accuracy',

        tf.keras.metrics.Precision(
            name='precision'
        ),

        tf.keras.metrics.Recall(
            name='recall'
        ),

        tf.keras.metrics.AUC(
            name='auc'
        )
    ]
)

In [ ]:
# ============================================================
# 13. Callbacks
# ============================================================
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ModelCheckpoint(
        'best_resnet50.keras',
        monitor='val_loss',
        save_best_only=False
    )
]

In [ ]:
# ============================================================
# 14. Train
# ============================================================
history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=callbacks,

    class_weight=class_weights
)

In [ ]:
# ============================================================
# 15. Plot history
# ============================================================
def plot_history(history):

    # Accuracy
    plt.figure(figsize=(8,5))

    plt.plot(
        history.history['accuracy'],
        label='Train Accuracy'
    )

    plt.plot(
        history.history['val_accuracy'],
        label='Validation Accuracy'
    )

    plt.title('ResNet50 Accuracy')

    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    plt.legend()

    plt.show()

    # Loss
    plt.figure(figsize=(8,5))

    plt.plot(
        history.history['loss'],
        label='Train Loss'
    )

    plt.plot(
        history.history['val_loss'],
        label='Validation Loss'
    )

    plt.title('ResNet50 Loss')

    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.legend()

    plt.show()

plot_history(history)


In [ ]:
# ============================================================
# 16. TEST TIME AUGMENTATION
# ============================================================
def tta_predict(model, images, n=2):

    preds = []

    for _ in range(n):

        augmented = data_augmentation(
            images,
            training=True
        )

        pred = model.predict(
            augmented,
            verbose=0
        )

        preds.append(pred)

    return np.mean(
        preds,
        axis=0
    )

In [ ]:
# ============================================================
# 17. Predictions
# ============================================================
def get_predictions(model, dataset):

    y_true = []
    y_pred = []

    for x, y in dataset:

        preds = tta_predict(
            model,
            x,
            n=5
        )

        y_true.extend(y.numpy())

        y_pred.extend(
            preds.flatten()
        )

    return np.array(y_true), np.array(y_pred)

y_true, y_pred = get_predictions(
    model,
    test_ds
)

In [ ]:
# ============================================================
# 18. Threshold optimization
# ============================================================
thresholds = np.arange(
    0.1,
    0.9,
    0.01
)

best_threshold = 0
best_f1 = 0

for t in thresholds:

    preds = (
        y_pred > t
    ).astype(int)

    f1 = f1_score(
        y_true,
        preds
    )

    if f1 > best_f1:

        best_f1 = f1

        best_threshold = t

print("\nBEST THRESHOLD:", best_threshold)
print("BEST F1:", best_f1)

# ============================================================
# 19. Final predictions
# ============================================================
y_pred_bin = (
    y_pred > best_threshold
).astype(int)


In [ ]:
# ============================================================
# 20. Classification report
# ============================================================
print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        y_true,
        y_pred_bin,
        target_names=[
            'NORMAL',
            'PNEUMONIA'
        ],
        digits=3
    )
)

# ============================================================
# 21. Confusion matrix
# ============================================================
cm = confusion_matrix(
    y_true,
    y_pred_bin
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=[
        'NORMAL',
        'PNEUMONIA'
    ],
    yticklabels=[
        'NORMAL',
        'PNEUMONIA'
    ]
)

plt.title(
    'ResNet50 Confusion Matrix'
)

plt.xlabel('Predicted')
plt.ylabel('True')

plt.show()

# ============================================================
# 22. ROC Curve
# ============================================================
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_pred
)

roc_auc = auc(
    fpr,
    tpr
)

plt.figure(figsize=(6,5))

plt.plot(
    fpr,
    tpr,
    label=f'AUC = {roc_auc:.3f}'
)

plt.plot([0,1], [0,1], '--')

plt.title('ResNet50 ROC Curve')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend()

plt.show()

print(f"\nAUC SCORE: {roc_auc:.4f}")

In [ ]:
# ============================================================
# SAVE MODEL + HISTORY
# ============================================================

import os
import json
import pandas as pd

# ============================================================
# DIRECTORIES
# ============================================================

PROJECT_DIR = '/content/drive/MyDrive/ChestXRayProject'

MODEL_DIR = os.path.join(PROJECT_DIR, 'models')

HISTORY_DIR = os.path.join(PROJECT_DIR, 'histories')

# Create folders automatically
os.makedirs(MODEL_DIR, exist_ok=True)

os.makedirs(HISTORY_DIR, exist_ok=True)

# ============================================================
# MODEL NAME
# ============================================================

MODEL_NAME = 'ResNet50'

# ============================================================
# SAVE MODEL
# ============================================================

model.save(
    f'{MODEL_DIR}/{MODEL_NAME}.keras'
)

print(f'Model saved in: {MODEL_DIR}/{MODEL_NAME}.keras')

# ============================================================
# SAVE HISTORY AS CSV
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_df.to_csv(
    f'{HISTORY_DIR}/{MODEL_NAME}_history.csv',
    index=False
)

print(f'History CSV saved in: {HISTORY_DIR}/{MODEL_NAME}_history.csv')

# ============================================================
# SAVE HISTORY AS JSON
# ============================================================

with open(
    f'{HISTORY_DIR}/{MODEL_NAME}_history.json',
    'w'
) as f:

    json.dump(
        history.history,
        f
    )

print(f'History JSON saved in: {HISTORY_DIR}/{MODEL_NAME}_history.json')